**Note:**

- Upgrade the matplotlib version form 3.2.2 to 3.5.3
  - Use command

        !pip install matplotlib --upgrade
- Restart the session

#1. Defining Problem Statement and Analysing basic metrics (10 Points)

##Download the dataset and observe a subset of the data

**Importing Required Libraries**

In [1]:
# !pip install matplotlib==3.5.3
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import seaborn as sns
import textwrap
import math

**Download the Netflix DataSet**

In [2]:
!gdown https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/001/125/original/aerofit_treadmill.csv?1639992749 -O aerofit_treadmill.csv
df = pd.read_csv('aerofit_treadmill.csv')

Downloading...
From: https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/001/125/original/aerofit_treadmill.csv?1639992749
To: /content/aerofit_treadmill.csv
100% 7.28k/7.28k [00:00<00:00, 14.3MB/s]


In [3]:
print(df.head())

  Product  Age  Gender  Education MaritalStatus  Usage  Fitness  Income  Miles
0   KP281   18    Male         14        Single      3        4   29562    112
1   KP281   19    Male         15        Single      2        3   31836     75
2   KP281   19  Female         14     Partnered      4        3   30699     66
3   KP281   19    Male         12        Single      3        3   32973     85
4   KP281   20    Male         13     Partnered      4        2   35247     47


In [12]:
def summarize_df(df_, print_summary=True, properties_as_columns=True):
  # Shape and memory usage of DataFrame
  print(f'RangeIndex: {df_.shape[0]} entries; Data columns (total {df_.shape[1]} columns)')
  memory_used = df_.memory_usage().sum()/1024
  if memory_used > 1024*1024:
    memory_used =  f'{round(memory_used/1024/1024, 1)}+ GB'
  elif memory_used > 1024:
    memory_used =  f'{round(memory_used/1024, 1)}+ MB'
  else:
    memory_used =  f'{round(memory_used, 1)}+ KB'
  print(f'memory usage: {memory_used}\n')

  # Create an empty df with column names from original df
  df2 = pd.DataFrame(columns=[None]+df_.columns.to_list())

  # Add dtype
  property_ = ['dtype']
  for clm in df_.columns:
    property_.append(df[clm].dtype)
  df2 = df2.append(pd.DataFrame([property_], columns=df2.columns))

  # Add Missing Values Counts
  property_ = ['Missing Counts']
  for clm in df_.columns:
    property_.append(df[clm].isna().sum())
  df2 = df2.append(pd.DataFrame([property_], columns=df2.columns))

  # Add nUniques
  property_ = ['nUniques']
  for clm in df_.columns:
    property_.append(df[clm].nunique())
  df2 = df2.append(pd.DataFrame([property_], columns=df2.columns))

  # Add unique values
  property_ = ['Top 5 Unique Values']
  for clm in df_.columns:
    df1 = df[clm].value_counts().reset_index()
    df1['margin'] = df1[clm]*100/ df1[clm].sum()
    property_.append(', '.join([i for i in df1.apply(lambda x:f"{x['index']} ({math.floor(x['margin'])}%)", axis=1).iloc[:5]]))
  df2 = df2.append(pd.DataFrame([property_], columns=df2.columns))

  # Getting Numeric Variables Statistics
  df4 = pd.DataFrame(columns=df_.columns.to_list())
  df4 = df4.append(df_.describe()).drop('count').rename({
      '25%': 'Q1',
      '50%': 'Median',
      '75%': 'Q3'
  }).reset_index().set_index('index')
  df4 = df4.T
  df4['LW (1.5)'] = df4.apply(lambda x: max(x['min'], x['Q1'] - 1.5*(x['Q3']-x['Q1'])), axis=1)
  df4['UW (1.5)'] = df4.apply(lambda x: min(x['max'], x['Q3'] + 1.5*(x['Q3']-x['Q1'])), axis=1)

  df4['mean-3*std'] = df4.apply(lambda x: max(x['min'], x['mean'] - 3*x['std']), axis=1)
  df4['mean+3*std'] = df4.apply(lambda x: min(x['max'], x['mean'] + 3*x['std']), axis=1)

  lst_IQR_Outlier = []
  lst_std_Outlier = []
  for clm in df4.index:
    lst_IQR_Outlier.append(df_.loc[(df[clm]<df4.loc[clm,'LW (1.5)']) | (df[clm]>df4.loc[clm,'UW (1.5)'])].shape[0])
    lst_std_Outlier.append(df_.loc[(df[clm]<df4.loc[clm,'mean-3*std']) | (df[clm]>df4.loc[clm,'mean+3*std'])].shape[0])

  df4['Outlier Count (1.5*IQR)'] = lst_IQR_Outlier
  df4['Outlier Count (3*std)'] = lst_std_Outlier

  df4 = df4.round(1).T.reset_index().rename({'index': None}, axis=1)
  df2 = df2.append(df4)

  # Sort the columns acording to dtype
  df2 = df2.set_index(None).T.sort_values('dtype', ascending=False).round(1)
  df2 = df2[['dtype', 'Missing Counts', 'nUniques', 'Top 5 Unique Values','min','max', 
  'LW (1.5)', 'Q1', 'Median', 'Q3',   'UW (1.5)', 'Outlier Count (1.5*IQR)', 
  'mean-3*std',  'mean', 'std', 'mean+3*std', 'Outlier Count (3*std)']]

  if not properties_as_columns: df2 = df2.T
  if print_summary: print(df2) 

  return df2

!jupyter nbconvert --to html /content/20221208_Aerofit_Project.ipynb

In [13]:
df_summary = summarize_df(df, print_summary=False, properties_as_columns=False)
df_summary

RangeIndex: 180 entries; Data columns (total 9 columns)
memory usage: 12.8+ KB



,Product,Gender,MaritalStatus,Age,Education,Usage,Fitness,Income,Miles
dtype,object,object,object,int64,int64,int64,int64,int64,int64
Missing Counts,0,0,0,0,0,0,0,0,0
nUniques,3,2,2,32,8,6,5,62,37
Top 5 Unique Values,"KP281 (44%), KP481 (33%), KP781 (22%)","Male (57%), Female (42%)","Partnered (59%), Single (40%)","25.0 (13%), 23.0 (10%), 24.0 (6%), 26.0 (6%), ...","16.0 (47%), 14.0 (30%), 18.0 (12%), 15.0 (2%),...","3.0 (38%), 4.0 (28%), 2.0 (18%), 5.0 (9%), 6.0...","3.0 (53%), 5.0 (17%), 2.0 (14%), 4.0 (13%), 1....","45480.0 (7%), 52302.0 (5%), 46617.0 (4%), 5457...","85.0 (15%), 95.0 (6%), 66.0 (5%), 75.0 (5%), 4..."
min,NaN,NaN,NaN,18.0,12.0,2.0,1.0,29562.0,21.0
max,NaN,NaN,NaN,50.0,21.0,7.0,5.0,104581.0,360.0
LW (1.5),NaN,NaN,NaN,18.0,12.0,2.0,1.5,29562.0,21.0
Q1,NaN,NaN,NaN,24.0,14.0,3.0,3.0,44058.75,66.0
Median,NaN,NaN,NaN,26.0,16.0,3.0,3.0,50596.5,94.0
Q3,NaN,NaN,NaN,33.0,16.0,4.0,4.0,58668.0,114.75


In [20]:
df_summary.T

,dtype,Missing Counts,nUniques,Top 5 Unique Values,min,max,LW (1.5),Q1,Median,Q3,UW (1.5),Outlier Count (1.5*IQR),mean-3*std,mean,std,mean+3*std,Outlier Count (3*std)
Product,object,0,3,"KP281 (44%), KP481 (33%), KP781 (22%)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
Gender,object,0,2,"Male (57%), Female (42%)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
MaritalStatus,object,0,2,"Partnered (59%), Single (40%)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
Age,int64,0,32,"25.0 (13%), 23.0 (10%), 24.0 (6%), 26.0 (6%), ...",18.0,50.0,18.0,24.0,26.0,33.0,46.5,5,18.0,28.788889,6.943498,49.6,1
Education,int64,0,8,"16.0 (47%), 14.0 (30%), 18.0 (12%), 15.0 (2%),...",12.0,21.0,12.0,14.0,16.0,16.0,19.0,4,12.0,15.572222,1.617055,20.4,3
Usage,int64,0,6,"3.0 (38%), 4.0 (28%), 2.0 (18%), 5.0 (9%), 6.0...",2.0,7.0,2.0,3.0,3.0,4.0,5.5,9,2.0,3.455556,1.084797,6.7,2
Fitness,int64,0,5,"3.0 (53%), 5.0 (17%), 2.0 (14%), 4.0 (13%), 1....",1.0,5.0,1.5,3.0,3.0,4.0,5.0,2,1.0,3.311111,0.958869,5.0,0
Income,int64,0,62,"45480.0 (7%), 52302.0 (5%), 46617.0 (4%), 5457...",29562.0,104581.0,29562.0,44058.75,50596.5,58668.0,80581.9,19,29562.0,53719.577778,16506.684226,103239.6,3
Miles,int64,0,37,"85.0 (15%), 95.0 (6%), 66.0 (5%), 75.0 (5%), 4...",21.0,360.0,21.0,66.0,94.0,114.75,187.9,13,21.0,103.194444,51.863605,258.8,4


In [21]:
print(df_summary.T)

                dtype Missing Counts nUniques  \
Product        object              0        3   
Gender         object              0        2   
MaritalStatus  object              0        2   
Age             int64              0       32   
Education       int64              0        8   
Usage           int64              0        6   
Fitness         int64              0        5   
Income          int64              0       62   
Miles           int64              0       37   

                                             Top 5 Unique Values      min  \
Product                    KP281 (44%), KP481 (33%), KP781 (22%)      NaN   
Gender                                  Male (57%), Female (42%)      NaN   
MaritalStatus                      Partnered (59%), Single (40%)      NaN   
Age            25.0 (13%), 23.0 (10%), 24.0 (6%), 26.0 (6%), ...     18.0   
Education      16.0 (47%), 14.0 (30%), 18.0 (12%), 15.0 (2%),...     12.0   
Usage          3.0 (38%), 4.0 (28%), 2.0 (18%),

In [23]:
df_summary['Education']

dtype                                                                  int64
Missing Counts                                                             0
nUniques                                                                   8
Top 5 Unique Values        16.0 (47%), 14.0 (30%), 18.0 (12%), 15.0 (2%),...
min                                                                     12.0
max                                                                     21.0
LW (1.5)                                                                12.0
Q1                                                                      14.0
Median                                                                  16.0
Q3                                                                      16.0
UW (1.5)                                                                19.0
Outlier Count (1.5*IQR)                                                    4
mean-3*std                                                              12.0

# Testing

In [ ]:
# df_ = df

# df4 = pd.DataFrame(columns=df_.columns.to_list())

# df4 = df4.append(df_.describe()).drop('count').rename({
#     '25%': 'Q1',
#     '50%': 'Median',
#     '75%': 'Q3'
# }).reset_index().set_index('index')
# df4 = df4.T
# df4['LW (1.5)'] = df4.apply(lambda x: max(x['min'], x['Q1'] - 1.5*(x['Q3']-x['Q1'])), axis=1)
# df4['UW (1.5)'] = df4.apply(lambda x: min(x['max'], x['Q3'] + 1.5*(x['Q3']-x['Q1'])), axis=1)

# df4['mean-3*std'] = df4.apply(lambda x: max(x['min'], x['mean'] - 3*x['std']), axis=1)
# df4['mean+3*std'] = df4.apply(lambda x: min(x['max'], x['mean'] + 3*x['std']), axis=1)

# lst_IQR_Outlier = []
# lst_std_Outlier = []
# for clm in df4.index:
#   lst_IQR_Outlier.append(df_.loc[(df[clm]<df4.loc[clm,'LW (1.5)']) | (df[clm]>df4.loc[clm,'UW (1.5)'])].shape[0])
#   lst_std_Outlier.append(df_.loc[(df[clm]<df4.loc[clm,'mean-3*std']) | (df[clm]>df4.loc[clm,'mean+3*std'])].shape[0])

# df4['Outlier Count (1.5*IQR)'] = lst_IQR_Outlier
# df4['Outlier Count (3*std)'] = lst_std_Outlier

# df4.round(1).T.reset_index().rename({'index': None}, axis=1)

#2. Non-Graphical Analysis: Value counts and unique attributes ​​(10 Points)

#3. Visual Analysis - Univariate & Bivariate (30 Points)

#4. Missing Value & Outlier Detection (10 Points)

#5. Business Insights based on Non-Graphical and Visual Analysis (10 Points)

#6. Recommendations (10 Points)